<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 3 — dbt practical and consolidation

<small>Data Creators › Day 3 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

## 🗺️ Table of contents

| Section | What happens |
|:---|:---|
| [How to use this notebook](#how-to-use) | Start here |
| [Welcome and launch](#launch) | Agenda · how the day runs · the pipeline · the launch box · what to keep |
| [Walkthrough: Phase 1, sources and staging](#phase1) | S1–S8 on DuckDB, section by section, with what learners should see |
| [Walkthrough: Phase 2, intermediate, marts, tests, docs](#phase2) | C1–C10 · materialisations · the NAV test · docs |
| [Circulating](#circulate) | The three-point checklist · blockers and fixes |
| [The platform session](#platform) | Agenda for a guest · the instructor-led version · learners' questions |
| [Stretch: macros, snapshots, incremental, unit tests](#stretch) | For fast finishers, with cells |
| [Consolidation](#recap) | The recap table · Activity 9 · tomorrow's volunteers |
| [Questions from the room](#questions) | Live Q&A, answers, sketches |

<a id="how-to-use"></a>

## How to use this notebook

- This is the anchor for Day 3: the agenda, every concept with its diagram, the whiteboard scripts, the activity briefs and the pointers into the learners' challenges. Keep it on the projector; switch to a terminal or the browser when a 🖥️ DEMO box says so.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard blocks are folded: open one when you reach it, ask the questions in order, write what the room says, add what is missing.
- The last section, **Questions from the room**, is yours to type in during the day: a table for questions, an empty cell to answer with SQL or Python, and `quick_diagram(...)` to sketch a diagram inline before reaching for Excalidraw.
- Learners can be given this notebook as it is; nothing in it is instructor-only. The instructor-only timing notes are in `run-sheet.md` beside it.
- The walkthrough cells run the same project as the learners' challenge, on DuckDB in `demos/portfolio_analytics/`. Run `bash demos/portfolio_analytics/demo.sh stretch` once before the day; the cells then rebuild pieces on demand.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day3-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day3-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 3 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

<a id="launch"></a>

## Welcome and launch

### Today

| Time | Block | Type | Where |
|---|---|---|---|
| 09:00–10:00 | Challenge 9 — dbt practical: launch and first hour | Practical | Anchor › Launch box; learners in challenges/v1/02-dbt-pipeline |
| 10:00–10:15 | Break | Break |  |
| 10:15–11:00 | Continue dbt practical | Practical | Anchor › Walkthrough cells (for support) |
| 11:00–12:00 | Guest session: the platform | Talk | Anchor › Platform session (two diagrams, agenda, questions) |
| 12:00–13:00 | Lunch | Break |  |
| 13:00–14:15 | Complete and extend dbt practical | Practical | Anchor › Stretch section |
| 14:15–14:30 | Break | Break |  |
| 14:30–14:45 | Consolidation | Debrief | Anchor › Recap table |
| 14:45–14:55 | Test review in pairs | Pair activity | Anchor › Activity 9 card |
| 14:55–15:00 | Closing discussion | Debrief | Anchor › Before learners leave |

The rhythm changes: a short launch, then self-paced practice from the README, with the instructor walking the room. A platform session at 11:00 (guest or instructor-led). Recap and a pair exercise at the end.

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday Catalyst generated a dbt project for you. Could you have written it, and could you tell if it is wrong?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>dbt from the inside: sources, staging, intermediate, marts, tests, docs, on the Portfolio Analytics case, on Databricks.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Reviewing generated code needs the same knowledge as writing it; the practical builds that knowledge one section at a time.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Work the README in order: concept, task, hint, check, variation, solution. Build Phase 1 before lunch, Phase 2 after.</div></div></div>

![The Portfolio Analytics pipeline the learners build: five sources, staging, intermediate, three marts, tests](../common/diagrams/png/d3-pipeline.png)

- Six raw feeds ship as seeds; five are modelled; refdata is there to inspect and for freshness practice.
- Staging renames and casts, one model per source; intermediate does the joins; marts are the consumer-aligned facts and aggregates.
- The tests are promises from a contract made executable: not null, unique, a relationship, accepted values, NAV within one basis point, the rollup reconciles.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; dbt practical (challenge 02) &nbsp;·&nbsp; about 3 hours across the day</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">In the training repository: <code>challenges/v1/02-dbt-pipeline/README.md</code>.<br><br>
         <strong>Where to work.</strong> <code>cd challenges/v1/02-dbt-pipeline/dbt</code> then <code>dbt debug</code>. Every <code>dbt</code> command runs from that folder; <code>mise run challenge:verify 02</code> and <code>challenge:reset 02</code> run from anywhere.<br>
         <strong>First three commands.</strong> <code>dbt debug</code> (connection), <code>dbt seed</code> (the eight CSVs into <code>training_dev.training_&lt;you&gt;</code>), <code>dbt ls --resource-type model</code> (ten stubs).<br>
         <strong>Phase 1</strong> (about an hour): S1 inspect the raw data · S2 read the project · S3 sources and freshness · S4 <code>stg_holdings</code> · S5 the other four staging models · S6 views vs tables · S7 build the staging layer · S8 a staging test.<br>
         <strong>Phase 2:</strong> C1 the enrichment join · C2 supporting intermediates · C3 <code>agg_fund_exposure</code> · C4 <code>fct_portfolio_returns</code> · C5 generic tests · C6 the NAV reconciliation test · C7 <code>dbt build</code> · C8 docs and lineage · C9 the Databricks post-hook · C10 the capstone mart and its test.<br>
         <strong>Gate.</strong> <code>mise run challenge:verify 02</code>: seeds, runs, tests; green means solved. <strong>Never</strong> run <code>challenge:reset 02</code> to "start clean": it deletes authored work.<br>
         <strong>Keep a list</strong> of commands, configuration changes, questions and discoveries; the recap is built from it.
         </div></div>

<div style="border:2px solid #D9EAD3; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🗣️ SAY</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">“Attempt each section before you open its solution. The solution is for comparing, not for copying.”<br>“You can run the raw dbt commands from the project folder. If dbt says it cannot find your credentials, open a fresh terminal so mise loads the environment, or rerun mise run setup:databricks.”</div></div>

<a id="phase1"></a>

## Walkthrough: Phase 1, sources and staging

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the same project, on DuckDB, section by section</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Everything below is the learners' challenge 02 solution ported to DuckDB in <code>demos/portfolio_analytics/</code> (same model names, layers, tags and tests; the Databricks-only post-hook becomes a no-op macro). Use a cell when someone asks 'what should I see?'. The file <code>demos/portfolio_analytics/README.md</code> maps each cell to the README section.</div></div>

#### S1 — Inspect the raw data: what feeds the models

In [ ]:
!cd demos/portfolio_analytics && dbt seed --full-refresh 2>&1 | grep -E "INSERT|Done" 
sql("select * from training_demo.holdings_raw order by fund_cd, pos_dt limit 8", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# A holding: one fund, one instrument, one day, valued in its local currency. Note fund_cd, pos_dt, isin, qty, mkt_val_lc, ccy_cd and _extracted_at.

#### S2 — Read the project: layers, materialisations, tags

In [ ]:
!cd demos/portfolio_analytics && dbt ls --resource-type model --exclude tag:stretch
!sed -n "/^models:/,/^seeds:/p" demos/portfolio_analytics/dbt_project.yml | head -24
# Ten models in three layers. Layers are folders with +materialized and +tags: select a layer with --select tag:staging.

#### S3 — Sources and freshness: five lines per source table, because the fund master has no load timestamp

In [ ]:
!cd demos/portfolio_analytics && dbt source freshness 2>&1 | grep -E "freshness|pass|warn|error|Done" | head -12
# Five sources report; product_ref has no loaded_at_field and is skipped. Tighten error_after to 0 hours and it flips to error: the check working, not a bug.

#### S4 — The first staging model: rename, cast, coalesce a null ISIN into CASH_<ccy>, no joins

In [ ]:
!sed -n "1,30p" demos/portfolio_analytics/models/staging/stg_holdings.sql

In [ ]:
!cd demos/portfolio_analytics && dbt build --select stg_holdings 2>&1 | grep -E "OK|PASS|ERROR|Done"
sql("select fund_id, position_date, instrument_id, quantity, mv_local_ccy, local_ccy from training_demo.stg_holdings order by 1,2 limit 8", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# Same grain as the source. instrument_id is an ISIN or CASH_<ccy>, never null; quantities are typed decimals.

#### S5 to S7 — The remaining staging models, then the whole layer by tag

In [ ]:
!cd demos/portfolio_analytics && dbt build --select tag:staging 2>&1 | grep -E "OK created|PASS|FAIL|ERROR|Done"
sql("select * from training_demo.stg_prices order by instrument_id, price_date limit 8", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# stg_positions and stg_prices dedup to the latest extract per natural key with row_number(); staging never raises the row count.

![View, table, incremental, snapshot: where each is set and when to use it](../common/diagrams/png/d3-materialisations.png)

- Staging and intermediate as views: cheap, always fresh, nobody reads them directly.
- Marts as tables: a stable, fast handoff for consumers.
- Set per folder in dbt_project.yml; override per model with config(). Both are configuration, not SQL.

#### S8 — A staging test is a query that must return no rows

In [ ]:
!sed -n "1,20p" demos/portfolio_analytics/models/staging/schema.yml
!cd demos/portfolio_analytics && dbt test --select stg_holdings 2>&1 | grep -E "PASS|FAIL|Done"
# not_null and unique compile to select-the-violations queries. Zero rows is a pass.

<a id="phase2"></a>

## Walkthrough: Phase 2, intermediate, marts, tests, docs

#### C1 — The enrichment join: holdings × prices × FX, grain unchanged

In [ ]:
!sed -n "1,45p" demos/portfolio_analytics/models/intermediate/int_holdings_enriched.sql
# Left joins on (instrument, date) and (currency, GBP, date). Ask the room: what happens when the FX rate is missing? Here it defaults to 1.0 via coalesce; the running example on Day 2 chose to leave it null and fail the test. Both are decisions; the contract should say which.

#### C3, C4 — The marts: an aggregate by strategy and the daily-grain fact with a calculated NAV

In [ ]:
!cd demos/portfolio_analytics && dbt build --select +agg_fund_exposure +fct_portfolio_returns 2>&1 | grep -E "OK created|PASS|FAIL|ERROR|Done" | tail -8
sql("select fund_id, position_date, strategy_code, exposure_base_ccy, instrument_count from training_demo.agg_fund_exposure order by 1,2", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
sql("select fund_id, position_date, instrument_id, mv_local_ccy, fx_rate_to_base, mv_base_ccy, round(calc_nav_per_unit, 6) as calc_nav_per_unit from training_demo.fct_portfolio_returns order by 1,2", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# calc_nav_per_unit = total base-currency market value / 100,000,000 synthetic units. Holdings only; cash excluded; a teaching measure, not an official NAV.

#### C5, C6 — Generic tests on the marts, and a singular reconciliation: NAV within one basis point of the custodian

In [ ]:
!sed -n "1,40p" demos/portfolio_analytics/tests/assert_nav_within_tolerance.sql
sql("select c.fund_id, c.position_date, round(c.calc_nav_per_unit,4) as calc, cu.nav_per_unit as custodian, round((c.calc_nav_per_unit - cu.nav_per_unit)/cu.nav_per_unit*10000, 3) as variance_bps from training_demo.fct_portfolio_returns c join training_demo.seed_custodian_nav cu using (fund_id, position_date) order by 1,2", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# The seeds are built so this passes within 1bp. Change one holding value and it will not: that is the test earning its keep.

#### C7, C8 — The full build, then docs and lineage

In [ ]:
!cd demos/portfolio_analytics && dbt build --exclude tag:stretch 2>&1 | grep -E "Done"
!cd demos/portfolio_analytics && dbt docs generate 2>&1 | grep -E "Catalog|Done" | tail -2
print("open target/index.html with: cd demos/portfolio_analytics && dbt docs serve --host 127.0.0.1 --port 8081   (the --host matters: DBT_HOST is used for the Databricks host in the learners' profile)")

#### C9, C10 — The Databricks post-hook and the capstone mart

In [ ]:
!cat demos/portfolio_analytics/macros/optimize_mart.sql
# On Databricks: OPTIMIZE ... ZORDER BY (fund_id, position_date) after each mart, a Delta optimisation. Here: a no-op. A macro that dispatches on target.type is how one project runs on two warehouses.

In [ ]:
sql("select * from training_demo.agg_fund_daily_exposure order by 1,2", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
!sed -n "1,20p" demos/portfolio_analytics/tests/reconcile_daily_exposure.sql
# C10: the (fund, date) rollup must preserve the strategy-level total. Learners create both the mart and this test.

<a id="circulate"></a>

## Circulating

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 CIRCULATION CHECKLIST — three questions per learner</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">
         <strong>Sources and staging:</strong> can they say what one row means, name its key, and explain the types and units?<br>
         <strong>Joins and models:</strong> are the lookup keys unique on the right-hand side, do dates belong in the join, and what happens to unmatched rows?<br>
         <strong>Tests and output:</strong> can they show a value checked independently of the model that produced it?<br><br>
         <strong>Ask:</strong> "Could this join pass its tests and still double the reported value?" Check the row grain, right-side uniqueness, unmatched rows and an independent total. A successful build alone is insufficient.
         </div></div>

| Symptom | Cause | Fix |
|---|---|---|
| `auth_type: oauth is required when not using access token` | The shell has no `DBT_ACCESS_TOKEN`; mise did not load the environment | Open a fresh terminal, or `mise run setup:databricks`; do not re-authenticate blindly (the repo's Copilot instructions give a three-step check) |
| `dbt show --inline "select …"` errors with unexpected extra arguments | Running through `mise run challenge:dbt` word-splits the SQL | Run raw `dbt` from the project folder, or use `dbt show --select <model>` |
| `--select tag:staging` selects nothing | The `models:` block in `dbt_project.yml` is missing or mis-keyed | Compare with the shipped `dbt_project.yml`; the project name must match |
| Full build fails in C7 on a placeholder mart | The capstone mart is created in C10, not shipped | Build with `--exclude agg_fund_daily_exposure` until C10 |
| Python crashes on the first query that returns rows | pandas result conversion on this image | The shipped profile sets `_disable_pandas: true`; check it is still there |
| `dbt docs serve` cannot bind | `DBT_HOST` holds the Databricks host | `dbt docs serve --host 127.0.0.1 --port 8081` |
| A relationship test fails after a learner edits a key | Keys changed in staging without the fund master | Trace the key back to the source; staging must not invent keys |

<a id="platform"></a>

## The platform session

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Why does the paved path look the way it does?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>An hour from the platform team: Airflow on Astronomer, Databricks and Unity Catalog, contracts and the Marketplace, the vended repository, compute, operations.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Learners will create products in that repository next month; the decisions behind it are the context the practicals cannot give.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Agree the agenda with the guest; keep the questions list ready; or give the instructor-led version from the diagrams.</div></div></div>

**Agenda to agree with a guest (one line each):** where code goes (the vended repository and its data_products folder) · how Airflow is vended (Astro per team, blueprints in YAML, Databricks jobs) · the three catalog types and how a product reaches the domain catalog · data contracts (ODCS) and the Marketplace · compute (serverless by default) · operations onboarding, warranty and Datadog · what is still changing.

![Unity Catalog on the paved path: application, source-aligned and domain catalogs, and the Marketplace](../common/diagrams/png/d3-catalogs.png)

- Your workspace's application catalog is where you build and test; nothing is shared from it.
- After the data quality rules pass, a vended blueprint copies the product to the domain catalog; that is how you share.
- One source-aligned catalog for ingested feeds; some producing teams publish straight into it.
- Access: schema-level groups automated today, table-level by request, attribute-level later. The contract (ODCS, YAML) records it and seeds the Marketplace.

![From one central Airflow to a vended Astro deployment per team](../common/diagrams/png/d3-airflow-federation.png)

- Today's single instance runs about twelve thousand DAGs on one cluster; noisy neighbours are the daily problem.
- The path vends an Astro deployment and a Databricks workspace per team from a template; net-new jobs start there.
- Cross-team dependencies: publish an asset; a platform plugin forwards it through Kafka; no shared API tokens.
- Operations: onboarding with a criticality level, a warranty period, vended Datadog dashboards; non-production Airflow hibernates outside business hours.

<details><summary><strong>Instructor-led version</strong> (twenty-five minutes, if no guest comes)</summary>

1. **Airflow today and tomorrow.** One open-source Airflow 2.11 instance, about 12,000 live DAGs (15,000 total) on a very large Kubernetes cluster; the vendor's recommended maximum is about 2,000. Options weighed: many self-run clusters (too much to manage), the cloud provider's managed Airflow (scaling and recovery not good enough for tier-one data), Astronomer (chosen: recovery objectives met with a plugin then a managed feature, the largest open-source contributor, a CLI that runs everything locally, quick vending). Migration is a "messy middle" over months: new jobs go to Astro now, critical pipelines last. Airflow 2.11 is end of life, so vended instances are Airflow 3 with the new UI; learners live in the Airflow UI, not the Astro console.
2. **Dependencies across teams.** Airflow assets: publish one, downstream DAGs subscribe. Across many Airflows a platform plugin forwards asset events through a Kafka topic using centrally managed secrets, so no team shares an API token. A system-lineage view helps discovery; it is not a data lineage tool. The legacy scheduler still exists; integration is by API in both directions.
3. **Databricks and Unity Catalog.** Seven years of Databricks; the old metastore could not share. Unity Catalog holds data outside any workspace and binds catalogs to workspaces, one Unity per region. Three catalog types: application (one per workspace, your scratch pad and jump point), source-aligned (one; ingested feeds with minimal translation; some teams publish directly), domain (curation; domain data and consumer-aligned data). Access at schema level by group, automated; table level by request; attribute level later.
4. **Contracts and the vended repository.** Contracts in the Open Data Contract Standard (YAML), required to publish to a domain catalog, seeding the Marketplace; a CI check for them is coming. The repository: dot-files for linters and tooling; `data_products/` with blueprints (dependency checks, a Databricks run-now task), clusters and pools, shared assets; a data product is a Databricks Asset Bundle plus a dbt project with datasets, each dataset holding its dbt, its Airflow YAML and its job YAML; an Airflow task only triggers the Databricks job, which runs the dbt commands. A Catalyst skill pulls template updates into your repository; review the pull request because you will have customised things.
5. **Compute and cost.** Serverless by default; classic compute warms up and costs while idle. Health-check DAGs are vended but cost money; non-production Airflow hibernates outside business hours; cost reports are coming; teams now own patching and vulnerabilities in their platform.
6. **Questions rooms ask:** tiering (the platform is tier one; disaster recovery rules will be the creator's to write), can an application catalog be shared (no: write to it, the blueprint copies after quality rules), should source teams publish directly to the source-aligned catalog (increasingly yes), is the DDL migration tool staying (probably replaced by contract-driven deployment), consumer workspaces for business users (being designed).

</details>

<a id="stretch"></a>

## Stretch: macros, snapshots, incremental, unit tests

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">The README's Phase 2 is done. What does a production dbt project have that this one does not?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Macros that dispatch on the warehouse, a custom generic test, a snapshot (SCD type 2), an incremental mart, a unit test, an exposure.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>These are the pieces the platform's vended project uses, and what tomorrow's bitemporal macro builds on.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Run the cells; then ask Copilot or Claude to add the same to the learner project, with the Catalyst skill for unit tests.</div></div></div>

In [ ]:
!cat demos/portfolio_analytics/macros/to_base_ccy.sql
!cat demos/portfolio_analytics/tests/generic/test_non_negative.sql
# A macro names a repeated expression once; a generic test is a macro that returns the violating rows.

In [ ]:
!cd demos/portfolio_analytics && dbt snapshot 2>&1 | grep -E "snapshot|Done"
sql("select fund_cd, fund_name, strategy_code, is_active, dbt_valid_from, dbt_valid_to from training_demo.fund_master_snapshot order by 1", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")
# A snapshot keeps history: change a watched column in the seed, snapshot again, and the old row closes (dbt_valid_to) while a new one opens. Tomorrow adds the second clock.

In [ ]:
!cd demos/portfolio_analytics && dbt run --select agg_fund_daily_exposure_incremental 2>&1 | grep -E "incremental|Done"
!cd demos/portfolio_analytics && dbt run --select agg_fund_daily_exposure_incremental 2>&1 | grep -E "incremental|Done"
!sed -n "1,25p" demos/portfolio_analytics/models/stretch/agg_fund_daily_exposure_incremental.sql
# First run: full load. Second run: only dates newer than the table holds, replaced on the unique key (delete+insert) so a re-delivered day does not duplicate.

In [ ]:
!sed -n "/unit_tests:/,$p" demos/portfolio_analytics/models/stretch/stretch.yml
!cd demos/portfolio_analytics && dbt test --select stg_holdings,test_type:unit 2>&1 | grep -E "unit_test|PASS|FAIL|Done"
# A unit test runs the model SQL on made-up rows and compares with expected rows: no warehouse data, pure logic.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, in the learner project</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst adding-dbt-unit-test
Add a dbt unit test for stg_holdings covering: a null ISIN becomes CASH_&lt;ccy&gt;; fund codes are trimmed and upper-cased; a row with a null position date is dropped.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/adding-dbt-unit-test  (same text) — or simply: add a unit_tests entry for stg_holdings in a YAML file covering those three cases</pre></div></div>

<a id="recap"></a>

## Consolidation

<details open><summary><strong>🖊️ Whiteboard — what did we learn? (fill gaps from this list, do not lecture it)</strong> (ask, write what the room says, add what is missing)</summary>
<table style="width:100%;border-collapse:collapse;margin:8px 0;color:#1a1a1a"><tr><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Ask</th><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Expect, then add</th></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Models and conventions</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Staging and integration models; CG naming (stg_, int_, fct_, agg_; on Friday src_, dom_, rpt_); the SQL behind a transformation is a SELECT dbt wraps.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Dependencies and change</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>ref() and source() build the graph; a source or schema change is visible in lineage; dbt docs show it.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Testing and contracts</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Schema tests and singular tests are the contract's quality rules made executable; catalog information and descriptions come from the same YAML.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Bitemporal loading</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Tomorrow morning: the columns and one or two strategies, from a volunteer or guest. Questions to carry: what is business time vs system time? what happens on a correction?</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Deployment context</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Tomorrow afternoon: Harness and CI/CD, with screenshots for anyone without access.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Catalyst</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Generation, iteration, and the repository's data_products area where the code goes; 80 to 90% of the code from a good contract.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Airflow preview</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>YAML-configured blueprints are tomorrow's topic; today's dbt project is what they will run.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Execution and storage</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>dbt SQL runs on Databricks as Spark; files on S3 in Delta: columnar, with logs and manifests that give transactions; you can insert and delete, unlike plain Parquet.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Catalogue conventions</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Application, domain and source-aligned catalogs; publish via the domain catalog; the Marketplace is discovery.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Quality and materialisation</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Checks at project and model level; materialisation set per folder in dbt_project.yml or in the model file.</td></tr></table></details>

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 9 &nbsp;—&nbsp; Test review in pairs &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 10 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Swap one mapping rule and the test that guards it with the person next to you. The reviewer proposes a wrong result the pipeline could produce: a duplicated join, a missing dated FX rate, a changed unit. Decide together whether the test would catch it. Agree one improvement and write down the expected failure.</strong><br><br><strong>Produce:</strong><br>• One rule, its test, and a wrong result the test would or would not catch<br>• One improvement (a new test, a tightened test, or an independent check) and the failure it should produce<br><br><strong>Debrief questions:</strong><br>• Which wrong results would have passed every test in the project?<br>• What value did you check without using the model that produced it?</div></div>

<div style="border:2px solid #D9EAD3; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🗣️ SAY</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">“Two or three examples to the room: the wrong result, whether the test caught it, the improvement.”<br>“Save the project where it is; tomorrow Airflow runs it. Completing Phase 2 is not a prerequisite for tomorrow: the Airflow challenge ships the solved dbt project.”</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ The working dbt project, model and test results, one independent output check and the outstanding questions for orchestration, kept where they are.<br>☐ Each learner can trace one plain-English mapping through its model and its test.<br>☐ Two volunteers (or a guest) confirmed for tomorrow's bitemporal and deployment slots, with the brief.<br>☐ The Airflow baseline is ready: Databricks auth works, challenge 02's seeds are loaded, and Astro has been started once after setup.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>